# S2 - One leaf, water then H2O2, dark frames in the run

The unattended run for the biological verification and the automation figure. Each leaf is its
own control:

1. fill the chamber with water (pump), dark frames (shutter closed);
2. **baseline** in water: `EXPOSURE_S` frames, one dark frame after every `DARK_EVERY` light
   frames. The afterglow of mounting decays here, so the analysis uses only the last
   `BASELINE_WINDOW_H` of the baseline;
3. exchange to H2O2 (pump), **response** frames with the same interleaved darks;
4. dark frames at the end.

Same camera operating point and sensor temperature for every run (`slim.CAMERAS`). The dark
level under every light frame is interpolated in time between the dark frames of the same run,
so dark-signal non-uniformity and slow drifts cancel without a separate dark night.

**Controls** use this notebook with another `CONDITION` (`slim.CONDITIONS`): `water_control`
(exchange to fresh water: the switch and the time since wounding alone), `no_leaf` (empty
chamber, water then H2O2: reagent and chamber), `heat_killed` (dead tissue + H2O2).
For `no_leaf` look at the difference, not the ratio (the baseline is ~0).

**Second session** with the 50 mm f/0.95 pair: change `OPTICS` only, and run S1 again for the
sample-plane reference.

**Hardware**: the dark slider (UC2 board channels 1/2, `bp.DarkSlider`) is the shutter, the pumps
sit on channels 3/4 (`bp.Pumps`). Without them (`SHUTTER_OPEN` / `PUMP` = None) set
`DARKS = "ends"` (darks only at the start and the end; you cap the camera), and the notebook
asks you for the exchange, so plan `BASELINE_H` such that it falls into working hours. When it
waits for you, the dashboard says so: interrupt it (■) and run the "Operator done" cell.

## 1 Connect

In [ ]:
from pathlib import Path
import slim
bp = slim.bp

SIMULATE = True        # True: slim.Simulator stands in for ImSwitch, camera, shutter, pump, beta light
CAMERA = "toupcam"     # "toupcam" or "andor": the one operating point of slim.CAMERAS
# None = $IMSWITCH_API_URL (ImSwitch's own notebooks). Andor rig e.g. "http://192.168.178.50", port 80
IMSWITCH_URL, PORT = "http://localhost", 8001

client = slim.Simulator(CAMERA) if SIMULATE else bp.connect(IMSWITCH_URL, PORT)
DATA_ROOT = slim.DATA_ROOT / "simulated" if SIMULATE else slim.DATA_ROOT
print("Connected to", client.base_uri, "| detectors:", client.settingsManager.getDetectorNames())
print("Operating point:", slim.CAMERAS[CAMERA]["settings"])

## 2 Settings

In [ ]:
RUN_NAME = "cress"
DRY_RUN = False
CONDITION = "leaf"          # "leaf", "water_control", "no_leaf", "heat_killed"
SAMPLE = {
    "id": "cress_01",       # one id per leaf, used by S3
    "species": "Lepidium sativum (garden cress)",
    "injury": "cut",
    "h2o2": "2 %",
    "prepared_at": None,    # ISO time of injury, e.g. "2026-11-10T16:40"
    "notes": "",
}
OPTICS = {"name": "4x/0.20 pair", "magnification": 1.0}   # session 2: {"name": "50/0.95 pair", ...}

EXPOSURE_S = 20 * 60
BASELINE_H = 3.0            # in water
RESPONSE_H = 6.0            # after the exchange
DARK_EVERY = 3              # light frames between two dark frames
DARKS = "interleaved"       # "ends" until the shutter moves by itself
START_DARKS = END_DARKS = 2
FILL_AT_START = True        # False if you fill the chamber with water by hand when mounting
FLUSH_S = 120               # pump time per exchange: >= 3 chamber volumes (measure with dye)
SETTLE_S = 300              # after the pump stops, before the next frame
BASELINE_WINDOW_H, RESPONSE_WINDOW_H = 1.5, 3.0
E_PER_ADU = None            # from S1 of this camera; S3 uses the latest S1 run anyway

# The shutter: the dark slider on channels 1/2 of the UC2 board, "open" = sample, "dark" = middle
# (timed: MIDDLE_S from the open stop). SHUTTER_OPEN = SHUTTER_CLOSE = None instead: the notebook
# asks you to cap the camera (then DARKS = "ends").
slider = bp.DarkSlider(client, travel_s=3.0, middle_s=1.0)
SHUTTER_OPEN, SHUTTER_CLOSE = slider.open, slider.dark

# The pumps on channels 3/4, one per reservoir. With PUMP = None the notebook asks you for the
# exchange; set PUMP = pumps.hooks() once the mapping below and FLUSH_S are checked with dye
# (pumps.run("water", 10) runs one by hand). A dry run with the pumps wired runs them FLUSH_S.
pumps = bp.Pumps(client, {"water": 3, "h2o2": 4})
PUMP = None

if DRY_RUN:
    EXPOSURE_S, START_DARKS, END_DARKS = 5, 1, 1
    BASELINE_H = RESPONSE_H = 4 * EXPOSURE_S / 3600
    FLUSH_S, SETTLE_S = 5, 0

c = slim.CONDITIONS[CONDITION]
STEPS = slim.leaf_schedule(EXPOSURE_S, BASELINE_H, RESPONSE_H, c["baseline"], c["response"],
                           DARK_EVERY, START_DARKS, END_DARKS, DARKS, FILL_AT_START)
print(f"{CONDITION}: {c['sample']}, {c['baseline']} -> {c['response']}")
print(slim.describe(STEPS))

## 3 Run folder, camera settings, environment log, cooldown, hardware

In [ ]:
try:
    if acq.running:
        raise RuntimeError("An acquisition is still running in this kernel - stop it first (Stop cell)")
except NameError:
    pass

run, env, detector, preset = slim.prepare(
    client, CAMERA, "S2_leaf_water_h2o2", RUN_NAME, dry_run=DRY_RUN, data_root=DATA_ROOT,
    wait=not (DRY_RUN or SIMULATE),
    kind="slim_leaf", title=f"S2 - {SAMPLE['id']}: {c['baseline']} then {c['response']}",
    description=f"{c['sample']} in the flow chamber, {c['baseline']} baseline, exchange to "
                f"{c['response']}, dark frames from the shutter during the run.",
    condition=CONDITION, sample={**SAMPLE, "mounted": c["sample"]}, optics=OPTICS, steps=STEPS,
    analysis={"baseline_window_h": BASELINE_WINDOW_H, "response_window_h": RESPONSE_WINDOW_H,
              "e_per_adu": E_PER_ADU},
    rois={"sample": None, "background": None})

if SIMULATE:
    shutter, fluidics, operator = client.shutter(run.log), client.fluidics(run.log), client.operator
    if CONDITION == "no_leaf":
        client.leaf = False
else:
    operator = slim.Operator(log=run.log, dry_run=DRY_RUN)
    shutter = (slim.Shutter("callable", SHUTTER_OPEN, SHUTTER_CLOSE, operator, log=run.log)
               if SHUTTER_OPEN else slim.Shutter("prompt", operator=operator, log=run.log))
    fluidics = (slim.Fluidics("callable", PUMP["select"], PUMP["pump_on"], PUMP["pump_off"],
                              FLUSH_S, SETTLE_S, operator, run.log)
                if PUMP else slim.Fluidics("prompt", flush_s=FLUSH_S, settle_s=SETTLE_S,
                                           operator=operator, log=run.log))
if DARKS == "interleaved" and not shutter.automatic and not DRY_RUN:
    raise RuntimeError("interleaved darks need a shutter that moves by itself: set "
                       "SHUTTER_OPEN / SHUTTER_CLOSE, or DARKS = 'ends'")
print(f"shutter: {shutter.mode}, pump: {fluidics.mode}")

## 4 Start

Mount the leaf, close the enclosure, switch the light off. Runs in a background thread and
returns at once.

In [ ]:
acq = slim.ProtocolAcquisition(run, client, detector, STEPS, shutter, fluidics, operator,
                               env=env, env_label=CAMERA, backend=preset["backend"]).start()

## 5 Dashboard

Redraws from the files every 15 s and shows when the run waits for you. Interrupting it (■) stops
only the display.

In [ ]:
slim.dashboard(run.dir, acq, operator, refresh_s=15)

## Operator done

Run this after doing what the dashboard asked, then restart the dashboard.

In [ ]:
operator.confirm()

## Stop

Interrupt the dashboard first (■). Cancels the running exposure; saved frames stay.

In [ ]:
acq.stop()
print(acq.state)

## Finish

In [ ]:
env.stop()
run.save_json("camera_end.json", bp.parameters(client, detector)[1])
run.log("run finished")

## Analysis

Works on any S2 run folder. Read off a **sample** ROI on the leaf (or the wound edge) and a
**background** ROI in the chamber next to it, `[x, y, width, height]` in pixels of the saved frame.
The background ROI measures the medium and the chamber, dark-subtracted the same way.

In [ ]:
RUN_DIR = run.dir      # or e.g. Path("../data/S2_leaf_water_h2o2/<run folder>")
bp.show_frame(RUN_DIR, index=-1);

In [ ]:
ROIS = client.rois if SIMULATE else {"sample": None, "background": None}   # e.g. {"sample": [80, 90, 60, 60], ...}
bp.set_rois(RUN_DIR, ROIS)
summary = slim.redraw(RUN_DIR, show=True)
for key in ("sample_baseline", "sample_response", "sample_ratio", "sample_ratio_err",
            "sample_difference", "sample_peak_after_exchange_h", "background_baseline",
            "background_response", "dark_drift_adu", "warnings"):
    print(f"{key:30s} {summary.get(key)}")

`denoising.py` (E5) works on these runs too: `dn.load_stack(RUN_DIR, label="response")`, with
the run's own dark frames (`label="dark"`).